
# Analisi `density_md_delta` vs Tg — Tg da MongoDB

Questo notebook:
- carica tre CSV separati con simulazioni MD a **250 K, 300 K e 350 K**;
- allinea i campioni tramite `polymer_id`;
- recupera la **Tg sperimentale** dalla collection MongoDB `biceranoPolymers`;
- calcola la variazione della densità tra 250 e 350 K;
- confronta densità assolute e variazione termica rispetto alla Tg;
- calcola correlazioni di Spearman e Pearson;
- produce grafici pronti per la relazione.


In [ ]:
from pathlib import Path

# Root del progetto, indipendente dalla directory di esecuzione del notebook.
PROJECT_NAME = "polymer_prediction"
ROOT = Path.cwd()
while ROOT != ROOT.parent and ROOT.name != PROJECT_NAME:
    ROOT = ROOT.parent
if ROOT.name != PROJECT_NAME:
    raise RuntimeError(f"Root del progetto {PROJECT_NAME!r} non trovata")

DATA_DIR = ROOT / "data" / "md_features"
OUTPUT_DIR = ROOT / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# CONFIGURAZIONE
# =========================

CSV_250 = DATA_DIR / "biceranoPolymers_analysis_summary_250.csv"
CSV_300 = DATA_DIR / "biceranoPolymers_analysis_summary_base.csv"
CSV_350 = DATA_DIR / "biceranoPolymers_analysis_summary_350.csv"

# Nome della colonna presente nei tre CSV che corrisponde a MongoDB _id
ID_COL = "polymer_id"

# Colonne della densità nei tre CSV
DENSITY_COLS = {
    250: "density_md_250",
    300: "density_md",
    350: "density_md_350",
}

SEP = ","

# MongoDB
MONGO_DB = "PolymerPrediction"
MONGO_COLLECTION = "biceranoPolymers"

In [ ]:

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dotenv import load_dotenv
from pymongo import MongoClient
from scipy.stats import spearmanr, pearsonr



## 1. Connessione a MongoDB e recupero della Tg

Il codice prova prima a leggere la Tg da `properties.Tg`; come fallback prova anche i campi top-level `Tg` e `tg`.


In [ ]:
load_dotenv(ROOT / ".env")

MONGO_URI = os.getenv("MONGO_URI")
if not MONGO_URI:
    raise RuntimeError(
        f"MONGO_URI non trovata. "
    )

client = MongoClient(MONGO_URI)
db = client[MONGO_DB]
collection = db[MONGO_COLLECTION]

print("MongoDB collegato:", MONGO_DB, "/", MONGO_COLLECTION)

In [ ]:

def extract_tg(doc):
    properties = doc.get("properties")

    if isinstance(properties, dict):
        value = properties.get("Tg")
        if value is not None:
            return value

    for key in ("Tg", "tg"):
        value = doc.get(key)
        if value is not None:
            return value

    return np.nan


mongo_rows = []

for doc in collection.find(
    {},
    {
        "_id": 1,
        "properties.Tg": 1,
        "Tg": 1,
        "tg": 1,
    }
):
    mongo_rows.append({
        ID_COL: str(doc["_id"]),
        "Tg": extract_tg(doc),
    })

targets = pd.DataFrame(mongo_rows)

targets["Tg"] = pd.to_numeric(targets["Tg"], errors="coerce")
targets = targets.dropna(subset=["Tg"])
targets = targets.drop_duplicates(subset=ID_COL, keep="first")

print("Documenti Mongo con Tg valida:", len(targets))
targets.head()


## 2. Caricamento dei tre CSV

In [ ]:

def load_simulation(path, temperature):
    df = pd.read_csv(path, sep=SEP)

    density_col = DENSITY_COLS[temperature]

    required = {ID_COL, density_col}
    missing = required - set(df.columns)

    if missing:
        raise ValueError(
            f"{path}: mancano le colonne {missing}.\n"
            f"Colonne disponibili: {list(df.columns)}"
        )

    out = df[[ID_COL, density_col]].copy()
    out[ID_COL] = out[ID_COL].astype(str)

    out = out.rename(
        columns={density_col: f"density_{temperature}"}
    )

    if out[ID_COL].duplicated().any():
        n_dup = out[ID_COL].duplicated().sum()
        print(
            f"ATTENZIONE: {path} contiene {n_dup} duplicati in {ID_COL}. "
            "Viene mantenuta la prima occorrenza."
        )
        out = out.drop_duplicates(subset=ID_COL, keep="first")

    return out


df250 = load_simulation(CSV_250, 250)
df300 = load_simulation(CSV_300, 300)
df350 = load_simulation(CSV_350, 350)

print("250 K:", df250.shape)
print("300 K:", df300.shape)
print("350 K:", df350.shape)


## 3. Controllo della copertura degli ID

In [ ]:

ids_250 = set(df250[ID_COL])
ids_300 = set(df300[ID_COL])
ids_350 = set(df350[ID_COL])
ids_tg = set(targets[ID_COL])

ids_all_sim = ids_250 & ids_300 & ids_350
ids_final = ids_all_sim & ids_tg

print("ID a 250 K:", len(ids_250))
print("ID a 300 K:", len(ids_300))
print("ID a 350 K:", len(ids_350))
print("ID presenti in tutte e tre le simulazioni:", len(ids_all_sim))
print("ID Mongo con Tg:", len(ids_tg))
print("ID finali simulazioni + Tg:", len(ids_final))

missing_tg = sorted(ids_all_sim - ids_tg)

if missing_tg:
    print("\nPrimi ID presenti nelle simulazioni ma senza Tg Mongo:")
    print(missing_tg[:10])


## 4. Merge delle simulazioni con la Tg da MongoDB

In [ ]:

merged = (
    df250
    .merge(df300, on=ID_COL, how="inner")
    .merge(df350, on=ID_COL, how="inner")
    .merge(targets, on=ID_COL, how="inner")
)

numeric_cols = [
    "density_250",
    "density_300",
    "density_350",
    "Tg",
]

for c in numeric_cols:
    merged[c] = pd.to_numeric(merged[c], errors="coerce")

merged = (
    merged
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=numeric_cols)
    .reset_index(drop=True)
)

print("Campioni finali utilizzabili:", len(merged))
merged.head()


## 5. Calcolo della variazione della densità

In [ ]:

merged["density_md_delta"] = (
    merged["density_350"] - merged["density_250"]
)

merged["density_md_delta_half"] = (
    merged["density_350"] - merged["density_250"]
) / 2.0

merged[
    [
        ID_COL,
        "density_250",
        "density_300",
        "density_350",
        "density_md_delta",
        "density_md_delta_half",
        "Tg",
    ]
].head()


## 6. Correlazioni con la Tg

In [ ]:

def correlation_summary(x_col):
    tmp = merged[[x_col, "Tg"]].dropna()

    rho_s, p_s = spearmanr(tmp[x_col], tmp["Tg"])
    r_p, p_p = pearsonr(tmp[x_col], tmp["Tg"])

    return pd.Series({
        "N": len(tmp),
        "Spearman rho": rho_s,
        "Spearman p": p_s,
        "Pearson r": r_p,
        "Pearson p": p_p,
    })


corr = pd.DataFrame({
    "density_250": correlation_summary("density_250"),
    "density_300": correlation_summary("density_300"),
    "density_350": correlation_summary("density_350"),
    "density_md_delta": correlation_summary("density_md_delta"),
    "density_md_delta_half": correlation_summary("density_md_delta_half"),
}).T

corr


## 7. Grafico principale: variazione della densità vs Tg

In [ ]:

x_col = "density_md_delta"

tmp = merged[[x_col, "Tg"]].dropna()
rho, p = spearmanr(tmp[x_col], tmp["Tg"])

fig, ax = plt.subplots(figsize=(7, 5))

ax.scatter(tmp[x_col], tmp["Tg"], alpha=0.8)

coef = np.polyfit(tmp[x_col], tmp["Tg"], 1)
x_line = np.linspace(tmp[x_col].min(), tmp[x_col].max(), 200)
y_line = coef[0] * x_line + coef[1]
ax.plot(x_line, y_line, linestyle="--")

ax.set_xlabel("Δ densità MD (350 K - 250 K) [g/cm³]")
ax.set_ylabel("Tg sperimentale [K]")
ax.set_title("Variazione della densità simulata vs Tg")

ax.text(
    0.03,
    0.97,
    f"Spearman ρ = {rho:.2f}\nN = {len(tmp)}",
    transform=ax.transAxes,
    va="top",
    ha="left"
)

plt.tight_layout()
plt.show()

print(f"Spearman rho = {rho:.6f}")
print(f"p-value = {p:.6g}")


## 8. Confronto con le densità assolute

In [ ]:

for x_col, label in [
    ("density_250", "Densità MD a 250 K [g/cm³]"),
    ("density_300", "Densità MD a 300 K [g/cm³]"),
    ("density_350", "Densità MD a 350 K [g/cm³]"),
    ("density_md_delta", "Δ densità MD (350 K - 250 K) [kg/m³]"),
]:
    tmp = merged[[x_col, "Tg"]].dropna()
    rho, p = spearmanr(tmp[x_col], tmp["Tg"])

    fig, ax = plt.subplots(figsize=(6.5, 4.8))
    ax.scatter(tmp[x_col], tmp["Tg"], alpha=0.8)

    ax.set_xlabel(label)
    ax.set_ylabel("Tg sperimentale [K]")
    ax.set_title(f"{label} vs Tg")

    ax.text(
        0.03,
        0.97,
        f"Spearman ρ = {rho:.2f}\nN = {len(tmp)}",
        transform=ax.transAxes,
        va="top",
        ha="left"
    )

    plt.tight_layout()
    plt.show()


## 9. Controllo dei campioni estremi

In [ ]:

cols_show = [
    ID_COL,
    "density_250",
    "density_300",
    "density_350",
    "density_md_delta",
    "Tg",
]

extremes = pd.concat([
    merged.nsmallest(5, "density_md_delta")[cols_show],
    merged.nlargest(5, "density_md_delta")[cols_show],
])

extremes


## 10. Esportazione dei dati unificati

In [ ]:
OUTPUT_CSV = OUTPUT_DIR / "density_delta_vs_tg_merged.csv"

merged.to_csv(OUTPUT_CSV, index=False)

print(f"Salvato: {OUTPUT_CSV}")